# ⚠️ 테스트 전용 — 절대 제출하지 마세요 ⚠️

## Pseudo-labeling 실험 (탐구 목적만)

**이 노트북의 결과물(CSV)은 제출하면 안 됩니다.** DACON 규칙에 다음 문구가 있습니다:

> 위 예시 외에도 test 데이터셋이 모델 학습에 활용되는 경우에 Data leakage에 해당됨

Pseudo-labeling은 test 데이터의 **피처(행)를 학습 데이터에 직접 추가**하는 기법이라, 라벨이
모델 추정치(진짜 정답 아님)라도 이 규칙에 걸릴 수 있는 회색지대입니다. 명확한 예외 조항이
없는 한, **"애매하면 안 한다"** 원칙으로 접근합니다.

이 노트북은 **순수하게 "효과가 있는 기법인지" 궁금해서 로컬로만 확인**하는 용도입니다.
실제로 점수가 개선되더라도, 이 결과로 만든 파일은 제출 후보에 넣지 마세요.

## 방법
1. 기존 최고 모델(`donor_cat`)의 test 예측을 가져옴
2. 확신도 상위/하위 N%(절대 임계값 아닌 **상대적 기준** — 예전엔 절대값 0.9+ 기준으로 폐기했었음)를
   "거의 확실한 정답"으로 보고 가짜 라벨 부여
3. **fold-safe 평가**: 가짜 라벨 행은 모든 학습 fold에는 추가하되, 검증(validation) fold에는
   절대 포함 안 함 (가짜 라벨로 자기 자신을 검증하는 건 의미 없는 누수이므로)
4. baseline(가짜 라벨 없음) vs pseudo-label 추가 버전 OOF AUC 비교

---
**저장 위치(참고용, 제출 폴더 아님):** `experiment_history/2차/2차_pseudolabel_TEST_ONLY.ipynb`


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [ ]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
SEED = 42

# 가짜 라벨 채택 기준: 상대적 퍼센타일 (절대 임계값 아님)
PSEUDO_TOP_PCT = 0.03     # 확률 상위 3% -> 라벨 1로 채택
PSEUDO_BOTTOM_PCT = 0.03  # 확률 하위 3% -> 라벨 0으로 채택

EMBRYO_COUNT_COLS = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
    "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수","해동된 배아 수",
    "해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","동결 배아 사용 여부",
    "신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
CLIP_RULES = {"총 생성 배아 수":40,"수집된 신선 난자 수":40,"미세주입된 난자 수":45,
    "혼합된 난자 수":40,"저장된 배아 수":30,"배아 이식 경과일":7,"난자 혼합 경과일":7,
    "배아 해동 경과일":2,"난자 해동 경과일":1}
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 풍부한 피처셋 빌드 (donor_cat과 동일한 전처리)

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
y = train_raw[TARGET_COL].values.astype(int)


def safe_ratio(df, numerator, denominator, new_col):
    df = df.copy()
    can = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can.astype(int)
    df[new_col] = np.where(can, df[numerator] / df[denominator], np.nan)
    return df


def build_full_features(df):
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)

    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    patient_mid = {"만18-34세":31,"만35-37세":36,"만38-39세":38.5,"만40-42세":41,
                    "만43-44세":43.5,"만45-50세":47.5,"알 수 없음":np.nan}
    donor_mid = {"만20세 이하":20,"만21-25세":23,"만26-30세":28,"만31-35세":33,
                 "만36-40세":38,"만41-45세":43,"알 수 없음":np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = df["난자 기증자 나이"].map(donor_mid) if "난자 기증자 나이" in df.columns else pd.Series(np.nan, index=df.index)
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    if "특정 시술 유형" in df.columns:
        s = df["특정 시술 유형"].astype(str)
        for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
            safe = token.lower().replace(" ", "_")
            df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    if {"시술 당시 나이", "난자 출처"}.issubset(df.columns):
        df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)

    return df


train_feat = build_full_features(train_raw.drop(columns=["ID"]))
test_feat = build_full_features(test_raw_full.drop(columns=["ID"]))

X_train_full = train_feat.drop(columns=[TARGET_COL])
X_test_full = test_feat.copy()

SENTINEL = "정보없음"
obj_cols = X_train_full.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    X_train_full[col] = X_train_full[col].fillna(SENTINEL)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
    train_categories = sorted(set(X_train_full[col].astype(str).unique()) | {SENTINEL})
    X_train_full[col] = pd.Categorical(X_train_full[col].astype(str), categories=train_categories)
    X_test_full[col] = pd.Categorical(X_test_full[col].astype(str), categories=train_categories)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
X_test_full = X_test_full[X_train_full.columns]

print(f"풍부한 피처셋: train {X_train_full.shape}, test {X_test_full.shape}")


## 2. 가짜 라벨 선정 (상대적 퍼센타일 기준)

In [ ]:
# 기존 최고 test 예측 불러오기 (ecdf 스택 결과 우선, 없으면 donor_cat으로 폴백)
ECDF_TEST_PATH = BLEND_DIR / "ecdf_stack_test.npy"
DONOR_TEST_PATH = BLEND_DIR / "test_donor_cat.npy"

if ECDF_TEST_PATH.exists():
    source_test_pred = np.load(ECDF_TEST_PATH)
    print(f"가짜 라벨 소스: ecdf_stack_test.npy")
elif DONOR_TEST_PATH.exists():
    source_test_pred = np.load(DONOR_TEST_PATH)
    print(f"가짜 라벨 소스: test_donor_cat.npy")
else:
    raise FileNotFoundError("기존 test 예측 파일을 찾을 수 없습니다 — 경로 확인 필요")

n_test = len(source_test_pred)
top_thresh = np.quantile(source_test_pred, 1 - PSEUDO_TOP_PCT)
bottom_thresh = np.quantile(source_test_pred, PSEUDO_BOTTOM_PCT)

pseudo_pos_mask = source_test_pred >= top_thresh
pseudo_neg_mask = source_test_pred <= bottom_thresh

print(f"상위 {PSEUDO_TOP_PCT:.0%} 임계값: {top_thresh:.4f} -> {pseudo_pos_mask.sum()}개 행을 라벨 1로")
print(f"하위 {PSEUDO_BOTTOM_PCT:.0%} 임계값: {bottom_thresh:.4f} -> {pseudo_neg_mask.sum()}개 행을 라벨 0으로")

pseudo_idx = np.where(pseudo_pos_mask | pseudo_neg_mask)[0]
pseudo_y = np.where(pseudo_pos_mask[pseudo_idx], 1, 0)

X_pseudo = X_test_full.iloc[pseudo_idx].reset_index(drop=True)
print(f"\n총 가짜 라벨 행 수: {len(pseudo_idx)} (train 대비 {len(pseudo_idx)/len(y):.2%})")


## 3. Fold-safe 비교: baseline vs +pseudo-label

In [ ]:
def run_catboost_pseudo(X, y, X_extra=None, y_extra=None, seed=42, n_splits=N_SPLITS):
    """X_extra/y_extra(가짜 라벨)는 모든 학습 fold에 추가되지만, 검증 fold에는 절대 들어가지 않음."""
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    params = dict(loss_function="Logloss", eval_metric="AUC", iterations=1500, learning_rate=0.03,
                  depth=6, l2_leaf_reg=5, random_seed=seed, early_stopping_rounds=100,
                  allow_writing_files=False, verbose=False)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]

        if X_extra is not None:
            X_tr = pd.concat([X_tr, X_extra], axis=0, ignore_index=True)
            y_tr = np.concatenate([y_tr, y_extra])

        tp = Pool(X_tr, y_tr, cat_features=cat_idx)
        vp = Pool(X_va, y_va, cat_features=cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(tp, eval_set=vp, use_best_model=True)
        oof[val_idx] = m.predict_proba(vp)[:, 1]
    return oof


print("baseline (가짜 라벨 없음)...")
t0 = time.time()
oof_baseline = run_catboost_pseudo(X_train_full, y, seed=SEED)
print(f"  AUC={roc_auc_score(y, oof_baseline):.5f}  ({time.time()-t0:.0f}s)")

print("\n+pseudo-label...")
t0 = time.time()
oof_pseudo = run_catboost_pseudo(X_train_full, y, X_extra=X_pseudo, y_extra=pseudo_y, seed=SEED)
print(f"  AUC={roc_auc_score(y, oof_pseudo):.5f}  ({time.time()-t0:.0f}s)")

print(f"\n차이(+pseudo - baseline): {roc_auc_score(y, oof_pseudo) - roc_auc_score(y, oof_baseline):+.5f}")
print("(참고: donor_cat 단독시드 기준 baseline ≈ 0.74047)")


## 4. 해석 (그리고 다시 한번: 제출 금지)

- 개선되더라도, 위에서 설명한 규칙 회색지대 문제가 있으니 **이 결과로 만든 제출 파일은 사용하지 않습니다.**
- 만약 효과가 확실히 있다고 나온다면, 팀원들과 "이 기법이 규칙상 괜찮은지" 먼저 논의하고 결정하세요
  (애매하면 안 하는 게 안전 — 특히 수상권일 때)
- `PSEUDO_TOP_PCT`/`PSEUDO_BOTTOM_PCT`를 바꿔서 더 보수적으로(1%) 또는 더 적극적으로(5%) 시도해볼 수 있음
